[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yassermakram/tiny-llm-education/blob/main/stage1_b_word_reference.ipynb)

# Stage 1b: Zero-Layer Transformer (Word-Level) 🗺️
## target: Dialect Subway Lines (Bigram Transition Graphs)

Now that we understand how character transition metrics work, let's step up to the **Word Level**.

In a word-level language model, the vocabulary is composed of whole words.
Because we are predicting word-to-word transitions, training a full PyTorch model can take much longer due to the large vocabulary size. 

Since we already proved in Stage 1a that training a Zero-Layer network converges to the statistical bigram counts, we will **skip the training loop** and directly calculate the optimal matrices using direct counts!

### 💡 What we are showing:
We will map out the "subway lines" of the dialects. By filtering for the strongest transitions of core grammatical words (like 'الذي' in MSA vs 'اللي' in Masri), we can visualize the structural paths of each dialect as a directed graph.


In [ ]:
# 🛠️ Setup: Install dependencies if running on Google Colab
import sys
if 'google.colab' in sys.modules:
    !pip install -q plotly pandas datasets networkx


In [ ]:
# 📦 Fetching MSA and Masri text (Words)
from datasets import load_dataset
import re

print("Fetching MSA from Wikipedia and Masri from Tweets...")

# Stream Wikipedia for MSA
msa_stream = load_dataset("wikimedia/wikipedia", "20231101.ar", split="train", streaming=True)

# Load Arabic Tweets for real street Masri
tweets_ds = load_dataset("amgadhasan/arabic_tweets_dialects", split="train")
# Filter for Egyptian tweets only
eg_tweets = tweets_ds.filter(lambda x: x['dialect'] == 'EG')

def clean_msa(stream, max_chars=500000):
    text = ""
    for article in stream:
        cleaned = re.sub(r'\s+', ' ', article['text'])
        cleaned = re.sub(r'[^\s\u0621-\u064A]', '', cleaned)
        text += cleaned + " "
        if len(text) >= max_chars:
            break
    return text[:max_chars]

def clean_masri(tweets, max_chars=500000):
    text = ""
    for t in tweets:
        cleaned = re.sub(r'\s+', ' ', t['text'])
        # Remove english mentions/links
        cleaned = re.sub(r'[a-zA-Z0-9_@]+', '', cleaned)
        cleaned = re.sub(r'[^\s\u0621-\u064A]', '', cleaned)
        text += cleaned + " "
        if len(text) >= max_chars:
            break
    return text[:max_chars]

print("Collecting MSA...")
msa_text = clean_msa(msa_stream)
print("Collecting Masri...")
masri_text = clean_masri(eg_tweets)

# Tokenise into words. At the word level our "tokens" are whitespace-separated
# words, so we split the cleaned text into a list of words for bigram counting.
# Shape: msa_words/masri_words are 1-D lists of word strings -> [n_words]
msa_words = msa_text.split()
masri_words = masri_text.split()

print(f"Loaded {len(msa_text)} chars / {len(msa_words)} words of MSA "
      f"and {len(masri_text)} chars / {len(masri_words)} words of Masri.")


In [ ]:
# 🧮 Compute Bigram Counts & Transition Probabilities Directly
from collections import Counter
import pandas as pd

def compute_bigram_transitions(words, top_n=1000):
    # Find top vocabulary words
    word_counts = Counter(words)
    vocab = [w for w, c in word_counts.most_common(top_n)]
    vocab_set = set(vocab)
    
    # Calculate transitions
    transitions = Counter()
    for i in range(len(words) - 1):
        w1, w2 = words[i], words[i+1]
        if w1 in vocab_set and w2 in vocab_set:
            transitions[(w1, w2)] += 1
            
    # Build transition probability dictionary
    # For each w1, what is the probability of w2?
    w1_totals = Counter()
    for (w1, w2), count in transitions.items():
        w1_totals[w1] += count
        
    probs = {}
    for (w1, w2), count in transitions.items():
        probs[(w1, w2)] = count / w1_totals[w1]
        
    return probs, vocab

print("Computing MSA bigrams...")
msa_probs, msa_vocab = compute_bigram_transitions(msa_words)
print("Computing Masri bigrams...")
masri_probs, masri_vocab = compute_bigram_transitions(masri_words)


## 🗺️ Building the Dialect Subway Map
We'll select key "seed" words that capture grammar or common starting points (like 'في', 'من', 'اللي', 'الذي', 'هو', 'هي') and trace the most probable next-word paths to draw a subway map of the language!


In [ ]:
# 🕸️ Plot the Weighted-Bet Transition Trees (MSA vs Masri)
# Each arrow points from a word to a likely NEXT word, and shows probability 3 ways:
#   • direction  — current word ──▶ next word
#   • THICKNESS  — thicker arrow = higher transition probability
#   • the number — the probability itself, printed on the arrow
# We grow a small tree from one seed word per dialect: at each step keep the
# top_k likeliest NEW next words, down to max_depth. No spring layout, and no
# magic probability threshold — top_k does the selecting.
from collections import defaultdict
import plotly.graph_objects as go
from plotly.subplots import make_subplots


def build_transition_tree(probs, seed, max_depth=2, top_k=3):
    """Grow a strict tree from `seed`.

    At each node keep the `top_k` likeliest next words we have not placed yet,
    expanding breadth-first to `max_depth`.
    Returns a list of edges as (parent, child, prob, parent_depth) tuples.
    """
    edges = []
    visited = {seed}
    frontier = [(seed, 0)]
    while frontier:
        node, node_depth = frontier.pop(0)
        if node_depth >= max_depth:
            continue
        outgoing = sorted(
            ((w2, p) for (w1, w2), p in probs.items() if w1 == node),
            key=lambda wp: wp[1], reverse=True,
        )
        kept = 0
        for child, p in outgoing:
            if child in visited:
                continue
            edges.append((node, child, p, node_depth))
            visited.add(child)
            frontier.append((child, node_depth + 1))
            kept += 1
            if kept >= top_k:
                break
    return edges


def tree_layout(edges, seed):
    """Deterministic left->right layout.

    x = depth (so the tree flows left to right); y = evenly spaced, vertically
    centred slot among the words at that depth. Returns {word: (x, y)}.
    """
    depth_of = {seed: 0}
    for _parent, child, _p, parent_depth in edges:
        depth_of[child] = parent_depth + 1
    levels = defaultdict(list)
    for word, d in depth_of.items():
        levels[d].append(word)
    pos = {}
    for d, words in levels.items():
        n = len(words)
        for i, word in enumerate(words):
            pos[word] = (d, (n - 1) / 2 - i)
    return pos


W_MIN, W_MAX = 1.5, 9.0


def _edge_width(p):
    """Map a probability in [0, 1] to a line width in [W_MIN, W_MAX]."""
    return W_MIN + p * (W_MAX - W_MIN)


def plot_weighted_tree(fig, row, edges, pos, color, seed):
    """Draw one weighted, directed tree into subplot `row` (1-indexed)."""
    suffix = "" if row == 1 else str(row)
    xref, yref = f"x{suffix}", f"y{suffix}"
    for parent, child, p, _depth in edges:
        x0, y0 = pos[parent]
        x1, y1 = pos[child]
        # the arrow itself carries direction + thickness
        fig.add_annotation(
            x=x1, y=y1, ax=x0, ay=y0,
            xref=xref, yref=yref, axref=xref, ayref=yref,
            showarrow=True, arrowhead=3, arrowsize=1,
            arrowwidth=_edge_width(p), arrowcolor=color,
            text="", standoff=16, startstandoff=22,
        )
        # the probability, printed at the arrow's midpoint
        fig.add_annotation(
            x=(x0 + x1) / 2, y=(y0 + y1) / 2,
            xref=xref, yref=yref, showarrow=False,
            text=f"{p:.2f}", font=dict(size=10, color="#333"),
            bgcolor="rgba(255,255,255,0.75)",
        )
    words = list(pos.keys())
    fig.add_trace(
        go.Scatter(
            x=[pos[w][0] for w in words],
            y=[pos[w][1] for w in words],
            mode="markers+text",
            text=words, textposition="middle center",
            textfont=dict(size=13, color="#111"),
            marker=dict(
                color=[color if w == seed else "#d9d9d9" for w in words],
                size=[42 if w == seed else 28 for w in words],
                line=dict(width=1.5, color="#333"),
            ),
            hoverinfo="text", showlegend=False,
        ),
        row=row, col=1,
    )
    fig.update_xaxes(showgrid=False, zeroline=False, showticklabels=False, row=row, col=1)
    fig.update_yaxes(showgrid=False, zeroline=False, showticklabels=False, row=row, col=1)


# --- Build & draw both dialects -------------------------------------------
MSA_SEED, MASRI_SEED = "الذي", "اللي"
MAX_DEPTH, TOP_K = 2, 3

msa_edges = build_transition_tree(msa_probs, MSA_SEED, MAX_DEPTH, TOP_K)
masri_edges = build_transition_tree(masri_probs, MASRI_SEED, MAX_DEPTH, TOP_K)
msa_pos = tree_layout(msa_edges, MSA_SEED)
masri_pos = tree_layout(masri_edges, MASRI_SEED)

fig = make_subplots(
    rows=2, cols=1, vertical_spacing=0.12,
    subplot_titles=(
        "MSA · seed الذي (relative pronoun)",
        "Masri · seed اللي (relative pronoun)",
    ),
)
plot_weighted_tree(fig, 1, msa_edges, msa_pos, "#2b6cb0", MSA_SEED)
plot_weighted_tree(fig, 2, masri_edges, masri_pos, "#dd6b20", MASRI_SEED)
fig.update_layout(
    height=680,
    title_text="Next-word bets: a thicker, labelled arrow = a likelier next word",
    margin=dict(l=20, r=20, t=80, b=20),
)
fig.show()